# EU AI Act & GDPR assistant on Google Colab

Runs the full pipeline on a Colab GPU (a free T4 is enough):

1. get the code and build the index (EUR-Lex download, chunking, bge-m3 embeddings);
2. serve an open 7B model with **vLLM** behind an OpenAI-compatible API;
3. ask questions with the three systems compared in the evaluation.

*Runtime → Change runtime type → T4 GPU* before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and dependencies

In [ ]:
import os
# Private repo: add a GitHub token (read access) as a Colab secret named GITHUB_TOKEN
# (key icon in the left sidebar). Not needed once the repo is public.
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
prefix = f"https://{token}@" if token else "https://"
REPO_URL = prefix + "github.com/carlosalomar/eu-ai-act-assistant.git"
if not os.path.exists("eu-ai-act-assistant"):
    !git clone -q {REPO_URL}
%cd eu-ai-act-assistant
!pip install -q -r requirements.txt vllm

## 2. Build the index
Downloads the official texts, splits them by article and paragraph and embeds them (about 2 minutes on a T4).

In [ ]:
!python -m ingest.download
!python -m ingest.parse
!python -m ingest.embed --batch-size 32

## 3. Serve the model with vLLM

`Qwen/Qwen2.5-7B-Instruct-AWQ` (4-bit) fits on a T4 and leaves room for the
embedding model and the reranker. Any instruction model works: change `MODEL`
(e.g. `meta-llama/Llama-3.1-8B-Instruct` on an A100, which needs a Hugging Face token).

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
os.environ["LLM_MODEL"] = MODEL
os.environ["LLM_BASE_URL"] = "http://localhost:8000/v1"
!nohup vllm serve {MODEL} --dtype half --max-model-len 8192 --gpu-memory-utilization 0.6 > vllm.log 2>&1 &

In [ ]:
import time, requests
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print("vLLM is up"); break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

## 4. Ask questions

In [ ]:
from rag.pipeline import Assistant

assistants = {mode: Assistant(mode) for mode in ("llm_only", "dense", "hybrid_rerank")}

def ask(question):
    for mode, assistant in assistants.items():
        answer = assistant.ask(question)
        print(f"=== {mode} | cited: {answer.cited_units}")
        print(answer.display_text, "\n")

ask("Which AI practices are prohibited?")

In [ ]:
ask("¿Un sistema de selección de personal con IA es de alto riesgo?")

In [ ]:
ask("Was regelt Artikel 22 der DSGVO?")

In [ ]:
ask("What is the corporate tax rate in Ireland?")  # not covered: should abstain

## 5. Run the evaluation

50 questions (20+ also in Spanish and German) × 3 systems. Generation and the
LLM judge are resumable: if Colab disconnects, re-run the cell and it continues.
Takes about 20-30 minutes on a T4.

In [ ]:
!python -m eval.run_eval --stage all

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/summary.md").read()))
display(Image("results/recall_at_5.png"))
display(Image("results/generation_metrics.png"))

Download the results to commit them (or push from Colab):

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")